In [29]:
import pandas as pd

data = {
    "order_id": [1, 2, 2, 3, 4, 5],
    "customer": ["Amit", "Riya", "Riya", None, "Karan", "Neha"],
    "amount": ["500", "700", "700", "abc", "1200", None],
    "city": ["Jaipur", "Delhi", "Delhi", "Jaipur", "jaipur", "Mumbai"],
}

df = pd.DataFrame(data)
print(df)
df.info()

   order_id customer amount    city
0         1     Amit    500  Jaipur
1         2     Riya    700   Delhi
2         2     Riya    700   Delhi
3         3     None    abc  Jaipur
4         4    Karan   1200  jaipur
5         5     Neha   None  Mumbai
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   order_id  6 non-null      int64 
 1   customer  5 non-null      object
 2   amount    5 non-null      object
 3   city      6 non-null      object
dtypes: int64(1), object(3)
memory usage: 324.0+ bytes


In [30]:
df=df.drop_duplicates()
print(df)


   order_id customer amount    city
0         1     Amit    500  Jaipur
1         2     Riya    700   Delhi
3         3     None    abc  Jaipur
4         4    Karan   1200  jaipur
5         5     Neha   None  Mumbai


In [31]:
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df.info()
print(df)

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 0 to 5
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  5 non-null      int64  
 1   customer  4 non-null      object 
 2   amount    3 non-null      float64
 3   city      5 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 200.0+ bytes
   order_id customer  amount    city
0         1     Amit   500.0  Jaipur
1         2     Riya   700.0   Delhi
3         3     None     NaN  Jaipur
4         4    Karan  1200.0  jaipur
5         5     Neha     NaN  Mumbai


In [32]:
df["city"] = df["city"].str.strip().str.title()
print(df["city"].value_counts())

print(df.isna().sum())

city
Jaipur    3
Delhi     1
Mumbai    1
Name: count, dtype: int64
order_id    0
customer    1
amount      2
city        0
dtype: int64


In [33]:
df["customer"] = df["customer"].fillna("Unknown")

df["amount_filled"] = df["amount"].isna()
df["amount"] = df["amount"].fillna(df["amount"].median())

print(df)

   order_id customer  amount    city  amount_filled
0         1     Amit   500.0  Jaipur          False
1         2     Riya   700.0   Delhi          False
3         3  Unknown   700.0  Jaipur           True
4         4    Karan  1200.0  Jaipur          False
5         5     Neha   700.0  Mumbai           True


In [34]:
import sqlite3

conn = sqlite3.connect("orders.db")
df.to_sql("orders", conn, if_exists="replace", index=False)

query = """
SELECT city,
       COUNT(*) AS num_orders,
       SUM(amount) AS total_amount
FROM orders
GROUP BY city
ORDER BY total_amount DESC
"""
print(pd.read_sql(query, conn))

     city  num_orders  total_amount
0  Jaipur           3        2400.0
1  Mumbai           1         700.0
2   Delhi           1         700.0


In [35]:
customers = pd.DataFrame({
    "customer": ["Amit", "Riya", "Karan", "Neha"],
    "tier": ["Gold", "Silver", "Gold", "Silver"],
})
customers.to_sql("customers", conn, if_exists="replace", index=False)

# HAVING: only cities with total sales above 1000
q1 = """
SELECT city, COUNT(*) AS num_orders, SUM(amount) AS total
FROM orders
GROUP BY city
HAVING SUM(amount) > 1000
"""
print(pd.read_sql(q1, conn))

# JOIN: total sales per customer tier
q2 = """
SELECT c.tier, COUNT(*) AS num_orders, SUM(o.amount) AS total
FROM orders o
LEFT JOIN customers c ON o.customer = c.customer
GROUP BY c.tier
"""
print(pd.read_sql(q2, conn))

     city  num_orders   total
0  Jaipur           3  2400.0
     tier  num_orders   total
0    None           1   700.0
1    Gold           2  1700.0
2  Silver           2  1400.0


In [36]:
import pandas as pd

def clean_orders(df):
    df = df.drop_duplicates()
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["city"] = df["city"].str.strip().str.title()
    df["customer"] = df["customer"].fillna("Unknown")
    df["amount_filled"] = df["amount"].isna()
    df["amount"] = df["amount"].fillna(df["amount"].median())
    return df

raw = pd.DataFrame({
    "order_id": [1, 2, 2, 3, 4, 5],
    "customer": ["Amit", "Riya", "Riya", None, "Karan", "Neha"],
    "amount": ["500", "700", "700", "abc", "1200", None],
    "city": ["Jaipur", "Delhi", "Delhi", "Jaipur", "jaipur", "Mumbai"],
})

cleaned = clean_orders(raw)
print(cleaned)

   order_id customer  amount    city  amount_filled
0         1     Amit   500.0  Jaipur          False
1         2     Riya   700.0   Delhi          False
3         3  Unknown   700.0  Jaipur           True
4         4    Karan  1200.0  Jaipur          False
5         5     Neha   700.0  Mumbai           True


/tmp/ipykernel_5810/1964261608.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
/tmp/ipykernel_5810/1964261608.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["city"] = df["city"].str.strip().str.title()
/tmp/ipykernel_5810/1964261608.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas

In [37]:
assert cleaned["order_id"].is_unique, "Duplicate order_ids found"
assert cleaned["amount"].isna().sum() == 0, "Missing amounts remain"
assert set(cleaned["city"]) == {"Jaipur", "Delhi", "Mumbai"}, "City names not cleaned"
assert len(cleaned) == 5, "Wrong number of rows"
print("All tests passed")

All tests passed


In [38]:
print(len(cleaned))
print(cleaned["order_id"].tolist())
print(raw.duplicated().sum())

5
[1, 2, 3, 4, 5]
1


In [39]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/clean", methods=["POST"])
def clean_endpoint():
    orders = request.get_json()
    df = pd.DataFrame(orders)
    cleaned_df = clean_orders(df)
    return jsonify(cleaned_df.to_dict(orient="records"))

In [40]:
client = app.test_client()

messy_orders = [
    {"order_id": 1, "customer": "Amit", "amount": "500", "city": "Jaipur"},
    {"order_id": 2, "customer": "Riya", "amount": "700", "city": "Delhi"},
    {"order_id": 2, "customer": "Riya", "amount": "700", "city": "Delhi"},
    {"order_id": 3, "customer": None, "amount": "abc", "city": "jaipur"},
]

response = client.post("/clean", json=messy_orders)
print(response.status_code)
print(response.get_json())

200
[{'amount': 500.0, 'amount_filled': False, 'city': 'Jaipur', 'customer': 'Amit', 'order_id': 1}, {'amount': 700.0, 'amount_filled': False, 'city': 'Delhi', 'customer': 'Riya', 'order_id': 2}, {'amount': 600.0, 'amount_filled': True, 'city': 'Jaipur', 'customer': 'Unknown', 'order_id': 3}]


/tmp/ipykernel_5810/1964261608.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
/tmp/ipykernel_5810/1964261608.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["city"] = df["city"].str.strip().str.title()
/tmp/ipykernel_5810/1964261608.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas

In [45]:
from flask import Flask, request, jsonify

app = Flask(__name__)   # a fresh app, with no requests handled yet


def summarize_orders(df):
    total = df["amount"].sum()
    top_city = df.groupby("city")["amount"].sum().idxmax()
    estimated = int(df["amount_filled"].sum())
    return (
        f"{len(df)} orders, total {total:.0f}. "
        f"Top city: {top_city}. "
        f"{estimated} amount(s) were estimated."
    )


@app.route("/clean", methods=["POST"])
def clean_endpoint():
    orders = request.get_json()
    df = clean_orders(pd.DataFrame(orders))
    return jsonify(df.to_dict(orient="records"))


@app.route("/clean-and-summarize", methods=["POST"])
def clean_and_summarize():
    try:
        orders = request.get_json()
        df = clean_orders(pd.DataFrame(orders))
        return jsonify({
            "summary": summarize_orders(df),
            "orders": df.to_dict(orient="records"),
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

In [46]:
client = app.test_client()

messy_orders = [
    {"order_id": 1, "customer": "Amit", "amount": "500", "city": "Jaipur"},
    {"order_id": 2, "customer": "Riya", "amount": "700", "city": "Delhi"},
    {"order_id": 2, "customer": "Riya", "amount": "700", "city": "Delhi"},
    {"order_id": 3, "customer": None, "amount": "abc", "city": "jaipur"},
]

response = client.post("/clean-and-summarize", json=messy_orders)
print(response.status_code)
print(response.get_json()["summary"])

200
3 orders, total 1800. Top city: Jaipur. 1 amount(s) were estimated.


In [47]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")
print("Key found, length:", len(key))

Key found, length: 53


In [51]:
import requests
from google.colab import userdata

def summarize_with_llm(df):
    fallback = summarize_orders(df)   # the plain summary you already built
    try:
        key = userdata.get("GEMINI_API_KEY")
        prompt = (
            "Write a 2-sentence business summary of these orders for a manager. "
            "Use ONLY the facts below. Do not add or calculate any other numbers.\n\n"
            f"Facts: {fallback}\n\n"
            "Table:\n" + df.to_string(index=False)
        )
        url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent"
        response = requests.post(
            url,
            headers={"x-goog-api-key": key},
            json={"contents": [{"parts": [{"text": prompt}]}]},
            timeout=15,
        )
        response.raise_for_status()
        text = response.json()["candidates"][0]["content"]["parts"][0]["text"]
        if not text.strip():
            return fallback
        return text.strip()
    except Exception as e:
        print("LLM failed, using fallback:", e)
        return fallback

print(summarize_with_llm(cleaned))

The business processed 5 orders totaling 3800, with Jaipur ranking as the top city. Within these transactions, 2 amounts were estimated.


In [49]:
import requests
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")
r = requests.get(
    "https://generativelanguage.googleapis.com/v1beta/models",
    headers={"x-goog-api-key": key},
    timeout=15,
)
print(r.status_code)
for m in r.json().get("models", []):
    if "generateContent" in m.get("supportedGenerationMethods", []):
        print(m["name"])

200
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
mode

In [52]:
app = Flask(__name__)   # fresh app


@app.route("/clean", methods=["POST"])
def clean_endpoint():
    orders = request.get_json()
    df = clean_orders(pd.DataFrame(orders))
    return jsonify(df.to_dict(orient="records"))


@app.route("/clean-and-summarize", methods=["POST"])
def clean_and_summarize():
    try:
        orders = request.get_json()
        df = clean_orders(pd.DataFrame(orders))
        return jsonify({
            "summary": summarize_with_llm(df),
            "orders": df.to_dict(orient="records"),
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

In [53]:
client = app.test_client()

messy_orders = [
    {"order_id": 1, "customer": "Amit", "amount": "500", "city": "Jaipur"},
    {"order_id": 2, "customer": "Riya", "amount": "700", "city": "Delhi"},
    {"order_id": 2, "customer": "Riya", "amount": "700", "city": "Delhi"},
    {"order_id": 3, "customer": None, "amount": "abc", "city": "jaipur"},
]

response = client.post("/clean-and-summarize", json=messy_orders)
print(response.status_code)
print(response.get_json()["summary"])

200
The business recorded 3 orders totaling 1800, with Jaipur identified as the top city. Additionally, 1 amount was estimated among these orders.
